# CN7 가스: 반대 방향 변화를 위한 크기·방향 분리
기존173 후보 고정, 동일 분할 유지. 학습 내부 선택과 사후 개발 최고를 구분한다.


In [1]:
import nbformat
old=nbformat.read("notebooks/170_cn7_gas_persistent_relation_experts.ipynb",as_version=4)
oldcode="\n".join(c.source for c in old.cells if c.cell_type=="code")
exec(oldcode[:oldcode.index("def predict(parent,q):")])
from sklearn.neighbors import NearestNeighbors
prev=nbformat.read("notebooks/171_cn7_gas_normal_relative_flow_experts.ipynb",as_version=4)
prevcode="\n".join(c.source for c in prev.cells if c.cell_type=="code")
exec(prevcode[prevcode.index("BASIC="):prevcode.index("def views(")])
VIEW_NAMES=["flow_absolute6","flow_direction_magnitude8","local_flow_absolute6","joint_magnitude4"]
controls=d[["Barrel_Temperature_5_raw","Hopper_Temperature_raw","Mold_Temperature_4_raw"]].to_numpy(float)
localraw=np.column_stack([raw[:,0],raw[:,2],raw[:,3],raw[:,4],raw[:,4]-raw[:,5],raw[:,6]])
localmemo={}
def local_z(ref,ids):
 key=(tuple(ref),tuple(ids))
 if key in localmemo:return localmemo[key]
 ni=ref[y[ref]==0]
 imp=SimpleImputer(keep_empty_features=True).fit(controls[ni]);cx=imp.transform(controls)
 sc=RobustScaler().fit(cx[ni]);cx=sc.transform(cx)
 imp=SimpleImputer(keep_empty_features=True).fit(localraw[ni]);rr=imp.transform(localraw)
 floor=[]
 for k in range(rr.shape[1]):
  dif=np.diff(np.unique(np.round(rr[ni,k],4)));nz=dif[dif>1e-8]
  step=max(float(np.min(nz)) if len(nz) else .01,1e-4)
  floor.append(max(step,float(np.std(rr[ni,k]))*.1))
 ans=np.empty((len(ids),rr.shape[1]))
 for run in pd.unique(g[ids]):
  normal=ni[g[ni]==run]
  if len(normal)<8:normal=ni
  pos=np.flatnonzero(g[ids]==run)
  knn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cx[normal])
  neighbors=knn.kneighbors(cx[ids[pos]],return_distance=False)
  for j,nn in zip(pos,neighbors):
   ns=normal[nn];ns=ns[ns!=ids[j]][:30]
   ans[j]=(rr[ids[j]]-np.median(rr[ns],axis=0))/np.maximum(rr[ns].std(axis=0),floor)
 localmemo[key]=np.clip(ans,-20,20)
 return localmemo[key]
def views(z,ctx,ids,ref):
 key=tuple(ref)
 if key not in scorememo:scorememo[key]=normal_scores(ref)
 ns=scorememo[key]
 mold3,mold4,speed,fill,pressure,switch,meter,plastic,cushion,bp,rpm=ns.T
 flow=np.column_stack([mold3,speed,fill,pressure,switch,pressure-speed])
 az=np.abs(flow)
 joint=np.column_stack([np.maximum(az[:,0],az[:,1]),np.minimum(az[:,1],az[:,2]),
  np.minimum(az[:,3],az[:,1]),np.mean(az,axis=1)])
 return {"flow_absolute6":az[ids],
 "flow_direction_magnitude8":np.column_stack([az[ids],flow[ids][:,[0,1]]]),
 "local_flow_absolute6":np.abs(local_z(ref,ids)),
 "joint_magnitude4":joint[ids]}
modelcode=oldcode[oldcode.index("def predict(parent,q):"):oldcode.index("records=[];choices=[]")]
modelcode=modelcode.replace("tr=views(z,ctx,ref);te=views(z,parent,q)","tr=views(z,ctx,ref,ref);te=views(z,parent,q,ref)")
exec(modelcode)
runner=oldcode[oldcode.index("records=[];choices=[]"):oldcode.index("# Descriptive missed-case audit")]
runner=runner.replace("cn7_170_persistence","cn7_175_magnitude")
runner=runner.replace('("add_to_165",pr|basepred)','("add_to_173",pr|basepred)')
needle="basepred=baseline.three_specialists_confirmed.to_numpy(int)"
replace="""basepred=baseline.three_specialists_confirmed.to_numpy(int)
p170=pd.read_csv(OUT/"cn7_170_persistence_predictions.csv",parse_dates=["TimeStamp"])
p170=p170[(p170.model=="persistent_abs4_ET")&(p170.rule=="normal95")].set_index("TimeStamp").loc[d.TimeStamp]
p173=pd.read_csv(OUT/"cn7_173_transfer_predictions.csv",parse_dates=["TimeStamp"])
p173=p173[(p173.model=="mechanism_joint8_LR")&(p173.rule=="inner_normal0.99")].set_index("TimeStamp").loc[d.TimeStamp]
basepred=basepred|p170.prediction.to_numpy(int)|p173.prediction.to_numpy(int)"""
runner=runner.replace(needle,replace)
exec(runner)
top=mm.sort_values(["F1","FP"],ascending=[False,True]).head(12)
primary=mm[mm.model.eq("inner_selected_mean3")]
best=top.iloc[0]
z=pp[(pp.model==best.model)&(pp.rule==best.rule)].set_index("TimeStamp").loc[d.TimeStamp]
pr=z.prediction.to_numpy(int)
if best.policy=="add_to_173":pr=pr|basepred
cases=d[["TimeStamp","gas","any_fail","run_id","seq"]].copy()
cases["baseline"]=basepred;cases["candidate"]=pr;cases["score"]=z.score.to_numpy();cases["threshold"]=z.threshold.to_numpy()
cases.to_csv(OUT/"cn7_175_magnitude_best_cases.csv",index=False,encoding="utf-8-sig")
print("\nTOP\n",top.to_string(index=False))
print("\nPRIMARY\n",primary.to_string(index=False))
print("\nGAS\n",cases[cases.gas].to_string(index=False))
print("\nNEW ALARMS\n",cases[cases.baseline.ne(cases.candidate)].to_string(index=False))
print("\nFULL COHORT",stats(y,pr))
lines=["# CN7 가스: 방향과 크기 분리/조건부 정상 대비 충전 편차","",
 "기준173: 8TP/5FP/F1.615. 새 전문가는 CN7 라벨만으로 학습하고 기준 앙상블에 추가한다.",
 "현재 충전6 절대편차, 절대편차6+방향2, 조건부 정상30개 대비 절대편차6, 동시강도4를 비교했다.",
 "조건부 정상 선택은 배럴5/호퍼/금형4 운전상태에 근거하며 학습 정상만 사용한다.",
 "RF/ET/LR 및 평균 앙상블. 같은20 시간블록/purge10/nested5.",
 "고정 최고는 평가 결과를 본 사후 개발 후보이며 독립 성능이 아니다.","",
 "## 고정 후보",top.to_string(index=False),"",
 "## 안쪽 선택",primary.to_string(index=False),"",
 "## 가스13개",cases[cases.gas].to_string(index=False),"",
 "## 새 경보",cases[cases.baseline.ne(cases.candidate)].to_string(index=False),"",
 "## 전체606샷",str(stats(y,pr))]
(ROOT/"reports/cn7_175_magnitude_review.md").write_text("\n".join(lines),encoding="utf-8")
(OUT/"cn7_175_magnitude_provenance.json").write_text(json.dumps({
 "input_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),"validation":"same20/purge10/nested5",
 "features":VIEW_NAMES,"conditional_controls":["Barrel_Temperature_5","Hopper_Temperature","Mold_Temperature_4"],
 "cutoffs":"inner normal95/97/99; no evaluation-label threshold fit",
 "caveat":"test-informed feature development and post-hoc fixed best",
 "base":"165 OR170ET95 OR173 jointLR99"},ensure_ascii=False,indent=2),encoding="utf-8")


Finished 1/20: cn7_R1__B00


Finished 2/20: cn7_R1__B01


Finished 3/20: cn7_R1__B02


Finished 4/20: cn7_R1__B03


Finished 5/20: cn7_R1__B04


...

Finished 18/20: cn7_R2__B07


Finished 19/20: cn7_R2__B08


Finished 20/20: cn7_R2__B09



TOP
                        model            rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1  AP_standalone
           flow_absolute6_LR        normal99 add_to_173 605   13   9   7   4 585   0.562500 0.692308 0.620690       0.184098
     local_flow_absolute6_RF        normal99 add_to_173 605   13   9   7   4 585   0.562500 0.692308 0.620690       0.014672
           flow_absolute6_ET        normal99 add_to_173 605   13   8   7   5 585   0.533333 0.615385 0.571429       0.012971
     local_flow_absolute6_LR        normal99 add_to_173 605   13   8   7   5 585   0.533333 0.615385 0.571429       0.184287
         joint_magnitude4_LR        normal99 add_to_173 605   13   8   8   5 584   0.500000 0.615385 0.551724       0.166712
  local_flow_absolute6_mean3        normal99 add_to_173 605   13   9  11   4 581   0.450000 0.692308 0.545455       0.020027
        inner_selected_mean3 inner_selection add_to_173 605   13   9  11   4 581   0.450000 0.692308 0.545455       0.0

557